# AI Toolkit by Ostris
## FLUX.1 Training


In [ ]:
!git clone https://github.com/ostris/ai-toolkit
# !mkdir -p /content/dataset

Cloning into 'ai-toolkit'...
remote: Enumerating objects: 5230, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 5230 (delta 64), reused 57 (delta 57), pack-reused 5158 (from 2)
Receiving objects: 100% (5230/5230), 30.44 MiB | 16.49 MiB/s, done.
Resolving deltas: 100% (3737/3737), done.


Put your image dataset in the `/content/dataset` folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cd ai-toolkit && git submodule update --init --recursive && pip install -r requirements.txt


Submodule 'repositories/batch_annotator' (https://github.com/ostris/batch-annotator) registered for path 'repositories/batch_annotator'
Submodule 'repositories/ipadapter' (https://github.com/tencent-ailab/IP-Adapter.git) registered for path 'repositories/ipadapter'
Submodule 'repositories/leco' (https://github.com/p1atdev/LECO) registered for path 'repositories/leco'
Submodule 'repositories/sd-scripts' (https://github.com/kohya-ss/sd-scripts.git) registered for path 'repositories/sd-scripts'
Cloning into '/content/ai-toolkit/repositories/batch_annotator'...
Cloning into '/content/ai-toolkit/repositories/ipadapter'...
Cloning into '/content/ai-toolkit/repositories/leco'...
Cloning into '/content/ai-toolkit/repositories/sd-scripts'...
Submodule path 'repositories/batch_annotator': checked out '420e142f6ad3cc14b3ea0500affc2c6c7e7544bf'
Submodule 'repositories/controlnet' (https://github.com/lllyasviel/ControlNet-v1-1-nightly.git) registered for path 'repositories/batch_annotator/repositor

## Model License
Training currently only works with FLUX.1-dev. Which means anything you train will inherit the non-commercial license. It is also a gated model, so you need to accept the license on HF before using it. Otherwise, this will fail. Here are the required steps to setup a license.

Sign into HF and accept the model access here [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)

[Get a READ key from huggingface](https://huggingface.co/settings/tokens/new?) and place it in the next cell after running it.

In [ ]:
import getpass
import os

# Prompt for the token
hf_token = getpass.getpass('Enter your HF access token and press enter: ')

# Set the environment variable v b
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable has been set.")

Enter your HF access token and press enter: ··········
HF_TOKEN environment variable has been set.


In [ ]:
import os
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Setup

This is your config. It is documented pretty well. Normally you would do this as a yaml file, but for colab, this will work. This will run as is without modification, but feel free to edit as you want.

In [ ]:
from collections import OrderedDict
job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        # this name will be the folder and filename name
        ('name', 'my_first_flux_lora_v2'),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                # root folder to save training sessions/samples/weights
                ('training_folder', '/content/drive/MyDrive/outputs'),
                # uncomment to see performance stats in the terminal every N steps
                #('performance_log_every', 1000),
                ('device', 'cuda:0'),
                # if a trigger word is specified, it will be added to captions of training data if it does not already exist
                # alternatively, in your captions you can add [trigger] and it will be replaced with the trigger word
                # ('trigger_word', 'image'),
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', 16),
                    ('linear_alpha', 16)
                ])),
                ('save', OrderedDict([
                    ('dtype', 'float16'),  # precision to save
                    ('save_every', 250),  # save every this many steps
                    ('max_step_saves_to_keep', 4)  # how many intermittent saves to keep
                ])),
                ('datasets', [
                    # datasets are a folder of images. captions need to be txt files with the same name as the image
                    # for instance image2.jpg and image2.txt. Only jpg, jpeg, and png are supported currently
                    # images will automatically be resized and bucketed into the resolution specified
                    OrderedDict([
                        ('folder_path', '/content/drive/MyDrive/floorplan_dataset_new/database/final_plans'),
                        ('caption_ext', 'txt'),
                        ('caption_dropout_rate', 0.05),  # will drop out the caption 5% of time
                        ('shuffle_tokens', False),  # shuffle caption order, split by commas
                        ('cache_latents_to_disk', True),  # leave this true unless you know what you're doing
                        ('resolution', [512, 768, 1024])  # flux enjoys multiple resolutions
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', 1),
                    ('steps', 4000),  # total number of steps to train 500 - 4000 is a good range
                    ('gradient_accumulation_steps', 1),
                    ('train_unet', True),
                    ('train_text_encoder', False),  # probably won't work with flux
                    ('content_or_style', 'balanced'),  # content, style, balanced
                    ('gradient_checkpointing', True),  # need the on unless you have a ton of vram
                    ('noise_scheduler', 'flowmatch'),  # for training only
                    ('optimizer', 'adamw8bit'),
                    ('lr', 4e-4),
                    # uncomment this to skip the pre training sample
                    #('skip_first_sample', True),

                    # ema will smooth out learning, but could slow it down. Recommended to leave on.
                    ('ema_config', OrderedDict([
                        ('use_ema', True),
                        ('ema_decay', 0.99)
                    ])),

                    # will probably need this if gpu supports it for flux, other dtypes may not work correctly
                    ('dtype', 'bf16')
                ])),
                ('model', OrderedDict([
                    # huggingface model name or path
                    ('name_or_path', 'black-forest-labs/FLUX.1-dev'),
                    ('is_flux', True),
                    ('quantize', True),  # run 8bit mixed precision
                    ('low_vram', True),  # uncomment this if the GPU is connected to your monitors. It will use less vram to quantize, but is slower.
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),  # must match train.noise_scheduler
                    ('sample_every', 250),  # sample every this many steps
                    ('width', 1024),
                    ('height', 1024),
                    ('prompts', [
                        '''Room Type: 3 bedrooms, 1 living room, 1 kitchen, 1 dining room, 1 bathroom, 1 laundry room.Room Location:
- Bedroom 1: top-left
- Bedroom 2: top-center
- Bedroom 3: bottom-left
- Living room: bottom-center
- Kitchen: bottom-right
- Dining room: top-right
- Bathroom: top-center
- Laundry room: bottom-center
Furnitures in Each Room:
- Bedroom 1: bed, sofa, lamp
- Bedroom 2: bed, sofa, lamp
- Bedroom 3: bed, sofa, lamp
- Living room: sofa, arm chair, coffee table
- Kitchen: table, chairs, refrigerator, stove, sink, dishwasher
- Dining room: table, chairs
- Bathroom: sink, toilet, shower
- Laundry room: washing machine, dryer
Design Features:
- The living room and dining room are connected by an open floor plan.
- The kitchen features a large island with a built-in sink and countertop.
- The bathroom has a walk-in shower with a glass door.
- The laundry room is located next to the kitchen for convenience.
Overall Description: This floor plan features a spacious and well-organized layout with separate rooms for each function. The open floor plan between the living room and dining room creates a sense of openness and connectivity. The kitchen is equipped with modern appliances and ample counter space, making it a functional area for cooking and entertaining. The bathroom and laundry room are designed for comfort and convenience, with the walk-in shower and large island in the kitchen being notable design features. Overall, this floor plan offers a balanced layout that caters to both functionality and aesthetics.'''
                        # 'a floor plan drawing, the rooms are used for the living room, kitchen, bedroom and bathroom,the furniture types are the chairs, the table, the sofa, the bed, the wardrobe, the wardrobe, the bed, the wardrobe, the bed, the wardrobe, the bed, the wardrobe, the bed, the wardrobe, the bed, the,'
                    ]),
                    ('neg', ''),  # not used on flux
                    ('seed', 42),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', 25)
                ]))
            ])
        ])
    ])),
    # you can add any additional meta info here. [name] is replaced with config name at top
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])


## Run it

Below does all the magic. Check your folders to the left. Items will be in output/LoRA/your_name_v1 In the samples folder, there are preiodic sampled. This doesnt work great with colab. They will be in /content/output

In [ ]:
run_job(job_to_run)


{
    "type": "sd_trainer",
    "training_folder": "/content/drive/MyDrive/outputs",
    "device": "cuda:0",
    "network": {
        "type": "lora",
        "linear": 16,
        "linear_alpha": 16
    },
    "save": {
        "dtype": "float16",
        "save_every": 250,
        "max_step_saves_to_keep": 4
    },
    "datasets": [
        {
            "folder_path": "/content/drive/MyDrive/floorplan_dataset_new/database/final_plans",
            "caption_ext": "txt",
            "caption_dropout_rate": 0.05,
            "shuffle_tokens": false,
            "cache_latents_to_disk": true,
            "resolution": [
                512,
                768,
                1024
            ]
        }
    ],
    "train": {
        "batch_size": 1,
        "steps": 4000,
        "gradient_accumulation_steps": 1,
        "train_unet": true,
        "train_text_encoder": false,
        "content_or_style": "balanced",
        "gradient_checkpointing": true,
        "noise_scheduler": "fl

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Quantizing transformer
Loading VAE
Loading T5


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Quantizing T5
Loading CLIP
Making pipe
Preparing Model
create LoRA network. base dim (rank): 16, alpha: 16
neuron dropout: p=None, rank dropout: p=None, module dropout: p=None
create LoRA for Text Encoder: 0 modules.
create LoRA for U-Net: 494 modules.
enable LoRA for U-Net
Dataset: /content/drive/MyDrive/floorplan_dataset_new/database/final_plans
  -  Preprocessing image dimensions


...

Saving at step 3500
Saved to /content/drive/MyDrive/outputs/my_first_flux_lora_v2/optimizer.pt
Removing old save: /content/drive/MyDrive/outputs/my_first_flux_lora_v2/my_first_flux_lora_v2_000002500.safetensors



my_first_flux_lora_v2:  94%|█████████▎| 3749/4000 [1:41:50<06:43,  1.61s/it, lr: 4.0e-04 loss: 1.001e-01]

Generating Images:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Images: 100%|██████████| 1/1 [00:22<00:00, 22.42s/it]

                                                                

Saving at step 3750
Saved to /content/drive/MyDrive/outputs/my_first_flux_lora_v2/optimizer.pt
Removing old save: /content/drive/MyDrive/outputs/my_first_flux_lora_v2/my_first_flux_lora_v2_000002750.safetensors



my_first_flux_lora_v2:  96%|█████████▌| 3842/4000 [1:44:22<04:28,  1.70s/it, lr: 4.0e-04 loss: 1.568e-01]

KeyboardInterrupt: 

## Done

Check your ourput dir and get your slider
